In [1]:
import requests
import io
import pandas
import numpy as np
#import matplotlib.pyplot as plt
from subprocess import Popen
import platform
import os
import shutil
from getpass import getpass
import netrc
from base64 import b64encode
from datetime import datetime

In [2]:
from tsgettoolbox import tsgettoolbox as tsget
from wdmtoolbox import wdmtoolbox as wdm


# Establish files required for API Access

In [3]:
# Set homeDir to current working directory
homeDir = os.path.expanduser("~/")
#homeDir = os.getcwd() + os.sep
# Create .urs_cookies and .dodsrc files in current directory
with open(homeDir + '.urs_cookies', 'w') as file:
    file.write('')
with open(homeDir + '.dodsrc', 'w') as file:
    file.write('HTTP.COOKIEJAR={}.urs_cookies\n'.format(homeDir))
    file.write('HTTP.NETRC={}.netrc'.format(homeDir))

print('Saved .urs_cookies and .dodsrc to:', homeDir)

# No need to copy .dodsrc since it's already in the working directory

Saved .urs_cookies and .dodsrc to: C:\Users\KVENABLE/


## Login with your Earthdata account. When completed your netrc file will be developed

In [4]:
urs = 'urs.earthdata.nasa.gov'    # Earthdata URL to call for authentication
prompts = ['Enter NASA Earthdata Login Username \n(or create an account at urs.earthdata.nasa.gov): ',
           'Enter NASA Earthdata Login Password: ']

with open(homeDir + '.netrc', 'w') as file:
    file.write('machine {} login {} password {}'.format(urs, getpass(prompt=prompts[0]), getpass(prompt=prompts[1])))
    file.close()

print('Saved .netrc to:', homeDir)

# Set appropriate permissions for Linux/macOS
if platform.system() != "Windows":
    Popen('chmod og-rw ~/.netrc', shell=True)

Saved .netrc to: C:\Users\KVENABLE/


## Authenticate Credentials from login to generate token

In [20]:
import importlib
import cloud_cover_timeseries_from_solar as ccts

importlib.reload(ccts)
cloud_cover_timeseries_from_solar = ccts.cloud_cover_timeseries_from_solar

test = cloud_cover_timeseries_from_solar(
    solr_values=[1000, 800],
    dates=df.index[:2].tolist(),
    lat_deg=lat_deg
)
print(type(test))

AttributeError: 'numpy.ndarray' object has no attribute 'index'

In [5]:
import importlib
import cloud_cover_timeseries_from_solar as ccts

importlib.reload(ccts)
cloud_cover_timeseries_from_solar = ccts.cloud_cover_timeseries_from_solar

def convertunitforHSPF(constituent, df, LDAS_var):
    '''This function is for unit conversion'''
    if constituent == "ATEMP": df = (df-273)
    #From K to degC
    if constituent == "SOLR":
        df = df * 0.0864


        constituent == "CLOU"
        print(f"Downloading SOLR for Cloud Cover from {LDAS_var}...")
        #print(type(df))
        #months = df.index.month
        #days = df.index.day
        # Apply the function
        clou_df = cloud_cover_timeseries_from_solar(
        solr_values=df.tolist(),
        dates=df.index.tolist(),
        lat_deg=lat_deg
        )
        print(type(clou_df))
        if hasattr(clou_df, "head"):
            print(clou_df.head())
            print(clou_df.info())
        else:
            print(df)
        df = clou_df["CLOU"] if "CLOU" in clou_df.columns else clou_df.squeeze()



        #solar_n_var = NLDAS_ConstituentDetails["CLOU"][2]
        #df = tsget.ldas(lat=station[1], lon=station[2],variables=solar_n_var, startDate=start_date, endDate=end_date)
        #df=clou_values
        #column_name = "CLOU"
        #print(f"Downloading SOLR for Cloud Cover from {LDAS_var}...")
        print(df.head())
        print(df.info())
        # Calculate CLOU values using the function
        #clou_ts = cloud_cover_timeseries_from_solar(
        #solr_values=df.tolist(),
        #dates=df['date'].tolist(),
        #lat_deg=lat_deg)
        # Add CLOU values to the DataFrame
        #df['CLOU'] = clou_ts.values
        #clou_df = pd.DataFrame({
        #'date': pd.to_datetime(clou_ts.dates),
        #'CLOU': clou_ts.values })
        #df = cloud_df
    #From W/m^2 to Langleys/day. 1 W/m^2 is 0.0864 Langleys/day
    if constituent == "PRECIP" and 'GLDAS' in LDAS_var:
        df = df * 3600
        #From kg/m^2/s to mm/hour
        #Assuming that 1 kg/m^2 is close to 1 mm
        df=df*3
            #GLDAS is 3 hourly data. THis changes values from
            #mm/hour to total precip for each times step of 3 hours.
    elif constituent == "PRECIP" and 'NLDAS' in LDAS_var:
        df=df
        #No change needed for NLDAS

    return df


In [21]:
import pandas as pd

def convertunitforHSPF(constituent, df, LDAS_var):
    """Unit conversion only. Always return a pandas Series with DateTimeIndex."""
    if isinstance(df, pd.DataFrame):
        df = df.iloc[:, 0]

    df = df.copy()
    df.index = pd.to_datetime(df.index)
    df = pd.to_numeric(df, errors="coerce")
    df = df.dropna()

    if constituent == "ATEMP":
        df = df - 273.15   # K to C

    elif constituent == "SOLR":
        df = df * 0.0864   # W/m^2 to langleys for your workflow
        df.name = "SOLR"

    elif constituent == "PRECIP" and "GLDAS" in LDAS_var:
        df = df * 3600 * 3

    elif constituent == "PRECIP" and "NLDAS" in LDAS_var:
        df = df

    return df


In [22]:
def build_clou_from_solr(solr_series, lat_deg):
    """Compute CLOU from SOLR and return a pandas Series."""
    solr_series = convertunitforHSPF("SOLR", solr_series, "SOLR")

    clou_df = cloud_cover_timeseries_from_solar(
        solr_values=solr_series.tolist(),
        dates=solr_series.index.to_pydatetime().tolist(),
        lat_deg=lat_deg
    )

    clou_series = clou_df["CLOU"].copy()
    clou_series.index = pd.to_datetime(clou_series.index)
    clou_series.name = "CLOU"
    return clou_series.dropna()

In [6]:
# Earthdata Login URL for obtaining the token, and creating one if it doesn't exist
url = 'https://urs.earthdata.nasa.gov/api/users/find_or_create_token'

# Earthdata Login credential prompts
#prompts = ['Enter NASA Earthdata Login Username \n(or create an account at urs.earthdata.nasa.gov): ',
 #          'Enter NASA Earthdata Login Password: ']

# Get credentials from user input
auth = netrc.netrc()
#auth = netrc.netrc(file=homeDir + '.netrc')
username, _, password = auth.authenticators('urs.earthdata.nasa.gov')

# Encode credentials using Base64
credentials = b64encode(f"{username}:{password}".encode('utf-8')).decode('utf-8')

# Headers with the Basic Authorization
headers = {
    'Authorization': f'Basic {credentials}'
}

# Make the POST request to get the token
response = requests.post(url, headers=headers)

# Check if the request was successful
if response.status_code == 200:
    # Parse the response JSON to get the token
    token_info = response.json()
    token = token_info.get("access_token")
    print("Token retrieved successfully")

    # Define the path for the .edl_token file in the home directory
    token_file_path = homeDir + ".edl_token"

    # Write the token to the .edl_token file
    with open(token_file_path, 'w') as token_file:
        token_file.write(token)

    print(f"Token saved to {token_file_path}")

else:
    print("Failed to retrieve token:", response.text)

Token retrieved successfully
Token saved to C:\Users\KVENABLE/.edl_token


In [7]:
help(convertunitforHSPF)

Help on function convertunitforHSPF in module __main__:

convertunitforHSPF(constituent, df, LDAS_var)
    This function is for unit conversion



In [16]:
def convertunitforHSPF(constituent, df, LDAS_var):
    '''This function is for unit conversion'''
    if constituent == "ATEMP": df = (df-273)
    #From K to degC
    if constituent == "SOLR":
        df = df * 0.0864
        # Calculate CLOU values using the function
        clou_ts = cloud_cover_timeseries_from_solar(
        solr_values=df.tolist(),
        dates=df.index.tolist(),
        lat_deg=lat_deg)
        # Add CLOU values to the DataFrame
        df= clou_ts.values
    #From W/m^2 to Langleys/day. 1 W/m^2 is 0.0864 Langleys/day
    if constituent == "PRECIP" and 'GLDAS' in LDAS_var:
        df = df * 3600
        #From kg/m^2/s to mm/hour
        #Assuming that 1 kg/m^2 is close to 1 mm
        df=df*3
            #GLDAS is 3 hourly data. THis changes values from
            #mm/hour to total precip for each times step of 3 hours.
    elif constituent == "PRECIP" and 'NLDAS' in LDAS_var:
        df=df
        #No change needed for NLDAS

    return df


In [9]:
import pandas as pd
import numpy as np

def process_cloud_cover(df):
    """
    Update the 'CLOU' column in the DataFrame based on the 'SOLR' values.
    Assumes df has a DateTimeIndex and columns 'SOLR' and 'CLOU'.
    """
    def calc_clou(solr):
        if solr > 990:
            return 0.0
        elif solr < 247.5:
            return 10.0
        else:
            return (((990.0 - solr) / 742.5) ** (1 / 3)) * 10.0

    df['CLOU'] = df['SOLR'].apply(calc_clou)
    return df

# Example usage:
# Create a DataFrame with DateTimeIndex and SOLR values
dates = pd.date_range('2026-03-01', periods=5, freq='D')
solr_values = [1000, 200, 500, 800, 300]
df = pd.DataFrame({'SOLR': solr_values, 'CLOU': np.nan}, index=dates)
# Process cloud cover
df = process_cloud_cover(df)
print(df)



            SOLR       CLOU
2026-03-01  1000   0.000000
2026-03-02   200  10.000000
2026-03-03   500   8.706292
2026-03-04   800   6.348713
2026-03-05   300   9.758526


In [11]:
import sys
import os
sys.path.append(os.getcwd())
from cloud_cover_timeseries_from_solar import cloud_cover_timeseries_from_solar

def process_cloud_cover_from_solr(df, solr_col='SOLR', lat_deg=40.0):
    """
    Update the 'CLOU' column in the DataFrame based on the SOLR (solar radiation) column.
    Assumes df has a DateTimeIndex and a column for solar radiation (Langleys).
    """
    df['CLOU'] = cloud_cover_timeseries_from_solar(df[solr_col], df.index, lat_deg=40.0)
    return df

# Example usage for SOLR to cloud cover using the imported function:
# dates = pd.date_range('2026-03-01', periods=5, freq='D')
# solr_values = [1000, 200, 500, 800, 300]
# lat_deg = 40.0
# df = pd.DataFrame({'SOLR': solr_values}, index=dates)
# df['CLOU'] = cloud_cover_timeseries_from_solar(df['SOLR'], df.index, lat_deg=lat_deg)
# print('CLOU TimeSeries:')
# for date, clou in zip(df.index, df['CLOU']):
#     print(f"Date: {date.date()} | CLOU: {clou:.2f}")



In [ ]:
import math

def nldas_dewpoint_timeseries_from_specific_humidity(aSpecHumidTSer, aAirTempTSer, aSource, aPressure=1013.25):
    """
    Compute dewpoint temperature timeseries from specific humidity and air temperature timeseries.
    Parameters:
        aSpecHumidTSer: Timeseries of specific humidity (kg/kg)
        aAirTempTSer: Timeseries of air temperature (deg F)
        aSource: Timeseries source object
        aPressure: Pressure in millibar (default: 1013.25)
    Returns:
        Dewpoint Temperature timeseries at the same times as aSpecHumidTSer (deg F)
    """
    lDewpointTS = atcTimeseries(aSource)
    copy_base_attributes(aSpecHumidTSer, lDewpointTS)
    lDewpointTS.Attributes['Constituent'] = 'DEWP'
    lDewpointTS.Attributes['TSTYPE'] = 'DEWP'
    lDewpointTS.Attributes['Scenario'] = 'COMPUTED'
    lDewpointTS.Attributes['Description'] = "Dewpoint temperature (Deg F) computed from Specific Humidity (kg/kg)"
    lDewpointTS.Attributes['SPCHUM'] = str(aSpecHumidTSer)
    lDewpointTS.Dates = aSpecHumidTSer.Dates
    lDewpointTS.numValues = aSpecHumidTSer.numValues

    lSpecHumidIndex = 0
    lAirTempIndex = 0
    lPoint = aSpecHumidTSer.Attributes.get('point', False)

    # Loop with zero-based index
    for lValueIndex in range(lDewpointTS.numValues):  # 0-based
        # Find matching indices
        lSpecHumidIndex = find_date_at_or_after(
            aSpecHumidTSer.Dates.Values,
            lDewpointTS.Dates.Value[lValueIndex],
            lSpecHumidIndex
        )
        lAirTempIndex = find_date_at_or_after(
            aAirTempTSer.Dates.Values,
            lDewpointTS.Dates.Value[lValueIndex],
            lAirTempIndex
        )

        # Handle point vs. interval logic with adjusted indices
        if lPoint:
            lDate = j2date(lDewpointTS.Dates.Value[lValueIndex])
        else:
            lDate = j2date(lDewpointTS.Dates.Value[lValueIndex - 1] if lValueIndex > 0 else lDewpointTS.Dates.Value[0])

        lAirTempC = (aAirTempTSer.Value[lAirTempIndex] - 32.0) * 5.0 / 9.0
        es = 6.112 * math.exp((17.67 * lAirTempC) / (lAirTempC + 243.5))
        e = aSpecHumidTSer.Value[lSpecHumidIndex] * aPressure / (0.378 * aSpecHumidTSer.Value[lSpecHumidIndex] + 0.622)
        RelHumid = e / es
        if RelHumid > 1.0:
            RelHumid = 1.0
        elif RelHumid < 0:
            RelHumid = 0.0
        lDewTempC = lAirTempC - ((100 - 100 * RelHumid) / 5)
        lDewpointTS.Value[lValueIndex] = (lDewTempC * 9.0 / 5.0) + 32.0

    return lDewpointTS



In [ ]:
#from datetime import datetime

def get_julian_day_from_date():
    """
    Prompt the user for a date and return the Julian day (day of year).
    """
    while True:
        date_str = input("Enter a date to calculate its Julian day (YYYY-MM-DD): ").strip()
        try:
            date_obj = datetime.strptime(date_str, "%Y-%m-%d")
            julian_day = date_obj.timetuple().tm_yday
            print(f"Julian day for {date_str} is: {julian_day}")

            #%%
            # Timeseries class for SOLR and CLOU
            class TimeSeries:
                def __init__(self, values, dates, constituent_name):
                    self.values = values  # list of floats
                    self.dates = dates    # list of datetime objects
                    self.constituent = constituent_name

                def process_clou_from_solr(solr_ts):
                    """
                    Calculate CLOU (cloud cover) from SOLR (solar radiation) timeseries.
                    Args:
                        solr_ts: TimeSeries object with .values (list of floats) and .dates (list of datetime)
                    Returns:
                        TimeSeries object for CLOU with same dates
                    """
                    clou_values = []
                for value in solr_ts.values:
                    if value > 990:
                        clou = 0.0
                    elif value < 247.5:
                        clou = 10.0
                    else:
                        clou = (((990.0 - value) / 742.5) ** (1 / 3)) * 10.0
                        clou_values.append(clou)
                return TimeSeries(clou_values, solr_ts.dates, constituent_name="CLOU")

                # Example usage
                from datetime import datetime
                    dates = [
                        datetime(2026, 3, 23),
                        datetime(2026, 3, 24),
                        datetime(2026, 3, 25),
                        datetime(2026, 3, 26),
                        datetime(2026, 3, 27)
                        ]
                    solr_values = [1000, 800, 500, 300, 200]  # Example Langley values
                    SOLR = TimeSeries(solr_values, dates, constituent_name="SOLR")
                    CLOU = process_clou_from_solr(SOLR)

                    print("CLOU (Cloud Cover) Results from SOLR:")
                            for date, solr, clou in zip(SOLR.dates, SOLR.values, CLOU.values):
                                print(f"Date: {date.date()} | SOLR: {solr} | CLOU: {clou:.2f}")
    lCloudTs.attributes["Scenario"] = "COMPUTED"
    lCloudTs.attributes["Description"] = "Daily Cloud Cover (0-10) computed from Daily Solar Radiation (Langleys)"
    lCloudTs.attributes.setdefault("History", []).append("Computed Daily Cloud Cover - inputs: DSOL, Latitude")
    lCloudTs.attributes["DSOL"] = str(aDSolTSer)
    lCloudTs.attributes["Latitude"] = aLatDeg
    lCloudTs.dates = aDSolTSer.dates
    lCloudTs.numValues = aDSolTSer.numValues

    # Prepare arrays: 0-based in Python
    num_vals = aDSolTSer.numValues
    lSolRad = list(aDSolTSer.values)[:num_vals]  # assuming aDSolTSer.values is at least length num_vals
    lCldCov = [0.0] * num_vals

    lPoint = aDSolTSer.attributes.get("point", False)
    for idx in range(num_vals):
        # Get date index, adjusting for 0-based Python vs 1-based VB
        if lPoint:
            lDate = J2Date(aDSolTSer.dates.value(idx))
        else:
            lDate = J2Date(aDSolTSer.dates.value(idx - 1)) if idx > 0 else J2Date(aDSolTSer.dates.value(0))
        # lDate: [year, month, day, ...]
        lCldCov[idx] = CloudCoverValueFromSolar(aLatDeg, lSolRad[idx], lDate[1], lDate[2])

    # Assign values back to timeseries, assuming lCloudTs.values is mutable and 0-based
    lCloudTs.values = lCldCov

    return lCloudTs

In [ ]:
def nldas_dewpoint_timeseries_from_specific_humidity(aSpecHumidTSer, aAirTempTSer, aSource, aPressure=1013.25):
    """
    Compute dewpoint temperature timeseries from specific humidity and air temperature timeseries.
    Parameters:
        aSpecHumidTSer: Timeseries of specific humidity (kg/kg)
        aAirTempTSer: Timeseries of air temperature (deg F)
        aSource: Timeseries source object
        aPressure: Pressure in millibar (default: 1013.25)
    Returns:
        Dewpoint Temperature timeseries at the same times as aSpecHumidTSer (deg F)
    """
    lDewpointTS = atcTimeseries(aSource)
    copy_base_attributes(aSpecHumidTSer, lDewpointTS)
    lDewpointTS.Attributes['Constituent'] = 'DEWP'
    lDewpointTS.Attributes['TSTYPE'] = 'DEWP'
    lDewpointTS.Attributes['Scenario'] = 'COMPUTED'
    lDewpointTS.Attributes['Description'] = "Dewpoint temperature (Deg F) computed from Specific Humidity (kg/kg)"
    lDewpointTS.Attributes['SPCHUM'] = str(aSpecHumidTSer)
    lDewpointTS.Dates = aSpecHumidTSer.Dates
    lDewpointTS.numValues = aSpecHumidTSer.numValues

    lSpecHumidIndex = 0
    lAirTempIndex = 0
    lPoint = aSpecHumidTSer.Attributes.get('point', False)

    # Loop with zero-based index
    for lValueIndex in range(lDewpointTS.numValues):  # 0-based
        # Find matching indices
        lSpecHumidIndex = find_date_at_or_after(
            aSpecHumidTSer.Dates.Values,
            lDewpointTS.Dates.Value[lValueIndex],
            lSpecHumidIndex
        )
        lAirTempIndex = find_date_at_or_after(
            aAirTempTSer.Dates.Values,
            lDewpointTS.Dates.Value[lValueIndex],
            lAirTempIndex
        )

        # Handle point vs. interval logic with adjusted indices
        if lPoint:
            lDate = j2date(lDewpointTS.Dates.Value[lValueIndex])
        else:
            lDate = j2date(lDewpointTS.Dates.Value[lValueIndex - 1] if lValueIndex > 0 else lDewpointTS.Dates.Value[0])

        lAirTempC = (aAirTempTSer.Value[lAirTempIndex] - 32.0) * 5.0 / 9.0
        es = 6.112 * math.exp((17.67 * lAirTempC) / (lAirTempC + 243.5))
        e = aSpecHumidTSer.Value[lSpecHumidIndex] * aPressure / (0.378 * aSpecHumidTSer.Value[lSpecHumidIndex] + 0.622)
        RelHumid = e / es
        if RelHumid > 1.0:
            RelHumid = 1.0
        elif RelHumid < 0:
            RelHumid = 0.0
        lDewTempC = lAirTempC - ((100 - 100 * RelHumid) / 5)
        lDewpointTS.Value[lValueIndex] = (lDewTempC * 9.0 / 5.0) + 32.0

    return lDewpointTS

In [ ]:
#from datetime import datetime

def get_julian_day_from_date():
    """
    Prompt the user for a date and return the Julian day (day of year).
    """
    while True:
        date_str = input("Enter a date to calculate its Julian day (YYYY-MM-DD): ").strip()
        try:
            date_obj = datetime.strptime(date_str, "%Y-%m-%d")
            julian_day = date_obj.timetuple().tm_yday
            print(f"Julian day for {date_str} is: {julian_day}")

            #%%
            # Timeseries class for SOLR and CLOU
            class TimeSeries:
                def __init__(self, values, dates, constituent_name):
                    self.values = values  # list of floats
                    self.dates = dates    # list of datetime objects
                    self.constituent = constituent_name

                def process_clou_from_solr(solr_ts):
                    """
                    Calculate CLOU (cloud cover) from SOLR (solar radiation) timeseries.
                    Args:
                        solr_ts: TimeSeries object with .values (list of floats) and .dates (list of datetime)
                    Returns:
                        TimeSeries object for CLOU with same dates
                    """
                    clou_values = []
                for value in solr_ts.values:
                    if value > 990:
                        clou = 0.0
                    elif value < 247.5:
                        clou = 10.0
                    else:
                        clou = (((990.0 - value) / 742.5) ** (1 / 3)) * 10.0
                        clou_values.append(clou)
                return TimeSeries(clou_values, solr_ts.dates, constituent_name="CLOU")

                # Example usage
                from datetime import datetime
                    dates = [
                        datetime(2026, 3, 23),
                        datetime(2026, 3, 24),
                        datetime(2026, 3, 25),
                        datetime(2026, 3, 26),
                        datetime(2026, 3, 27)
                        ]
                    solr_values = [1000, 800, 500, 300, 200]  # Example Langley values
                    SOLR = TimeSeries(solr_values, dates, constituent_name="SOLR")
                    CLOU = process_clou_from_solr(SOLR)

                    print("CLOU (Cloud Cover) Results from SOLR:")
                            for date, solr, clou in zip(SOLR.dates, SOLR.values, CLOU.values):
                                print(f"Date: {date.date()} | SOLR: {solr} | CLOU: {clou:.2f}")
    lCloudTs.attributes["Scenario"] = "COMPUTED"
    lCloudTs.attributes["Description"] = "Daily Cloud Cover (0-10) computed from Daily Solar Radiation (Langleys)"
    lCloudTs.attributes.setdefault("History", []).append("Computed Daily Cloud Cover - inputs: DSOL, Latitude")
    lCloudTs.attributes["DSOL"] = str(aDSolTSer)
    lCloudTs.attributes["Latitude"] = aLatDeg
    lCloudTs.dates = aDSolTSer.dates
    lCloudTs.numValues = aDSolTSer.numValues

    # Prepare arrays: 0-based in Python
    num_vals = aDSolTSer.numValues
    lSolRad = list(aDSolTSer.values)[:num_vals]  # assuming aDSolTSer.values is at least length num_vals
    lCldCov = [0.0] * num_vals

    lPoint = aDSolTSer.attributes.get("point", False)
    for idx in range(num_vals):
        # Get date index, adjusting for 0-based Python vs 1-based VB
        if lPoint:
            lDate = J2Date(aDSolTSer.dates.value(idx))
        else:
            lDate = J2Date(aDSolTSer.dates.value(idx - 1)) if idx > 0 else J2Date(aDSolTSer.dates.value(0))
        # lDate: [year, month, day, ...]
        lCldCov[idx] = CloudCoverValueFromSolar(aLatDeg, lSolRad[idx], lDate[1], lDate[2])

    # Assign values back to timeseries, assuming lCloudTs.values is mutable and 0-based
    lCloudTs.values = lCldCov

    return lCloudTs

In [ ]:
import numpy as np
import pandas as pd

def CloudCoverTimeseriesFromSolar(dsol_series, source, lat_deg):
    """
    Compute daily cloud cover based on daily solar radiation.

    Parameters
    ----------
    dsol_series : pandas.Series or pandas.DataFrame
        Daily solar radiation timeseries (values indexed by date).
    source : object
        Place-holder for data source, not used in this context.
    lat_deg : float
        Latitude in degrees.

    Returns
    -------
    pd.Series
        Daily cloud cover timeseries.
    """
    # Assume dsol_series is a pandas.Series indexed by date (datetime).
    # Make a copy for attributes if necessary.
    lCloudTs = pd.Series(index=dsol_series.index, dtype=float)

    # For each date, calculate cloud cover using CloudCoverValueFromSolar.
    for idx, (date, sol_rad) in enumerate(dsol_series.items()):
        # J2Date is unnecessary if "date" is already a pandas.Timestamp.
        month = date.month
        day = date.day
        lCloudTs[date] = CloudCoverValueFromSolar(lat_deg, sol_rad, month, day)

    return lCloudTs

# Stub for the CloudCoverValueFromSolar function.
def CloudCoverValueFromSolar(lat_deg, sol_rad, month, day):
    # The actual implementation will depend on the domain equations.
    # For this stub, we'll just return a dummy value (should be updated!).
    # Replace this logic with the real calculation.
    return np.nan  # TODO: Replace with actual formula

# Example usage:
# Suppose you have a daily solar radiation series like this:
dsol_series = pd.Series(
     data=[600, 700, 650, 752],
     index=pd.date_range(start='2025-01-01', periods=4)
)
cloud_cover_series = CloudCoverTimeseriesFromSolar(dsol_series, None, 38.0)

In [ ]:
print(cloud_cover_series)

In [17]:
Constituent=['ATEMP','PRECIP', 'SOLR', 'WIND', 'CLOU']
#Please refer to the GLDAS2 documentation below for more constituents.
#https://hydro1.gesdisc.eosdis.nasa.gov/data/GLDAS/README_GLDAS2.pdf
#https://hydro1.gesdisc.eosdis.nasa.gov/data/GLDAS/GLDAS_NOAH025_3H_EP.2.1/doc/README_GLDAS2.pdf


GLDAS_ConstituentDetails={
                    "PRECIP":["Precipitation","mm",
                    "GLDAS2:GLDAS_NOAH025_3H_2_1_Rainf_f_tavg","kg/m^2/s"],
                    "ATEMP":["Air Temperature","Fahrenheit",
                    "GLDAS2:GLDAS_NOAH025_3H_2_1_Tair_f_inst","K"],
                    "SOLR":["Shortwave radiation flux","Langleys",
                    "GLDAS2:GLDAS_NOAH025_3H_2_1_SWdown_f_tavg","W m-2"],
                    "WIND": ["Wind Speed", "m s-1",
                             "GLDAS2:GLDAS_NOAH025_3H_2_1_Wind_f_inst", "m s-1"], #average SW values for GLDAS. Instaneous is not available. I am using average values now.
                    "CLOU":["Cloud Cover Estimate","Cloud Fraction", "GLDAS2:GLDAS_NOAH025_3H_2_1_SWdown_f_tavg","W m-2"]
}
#Please refer to the NLDAS documentation below for more constituents.
#https://hydro1.gesdisc.eosdis.nasa.gov/data/NLDAS/NLDAS2_README.pdf
NLDAS_ConstituentDetails={
                    "PRECIP":["Precipitation","mm",
                    "NLDAS:NLDAS_FORA0125_H_2_0_Rainf","mm"],
                    "ATEMP":["Air Temperature","Fahrenheit",
                    "NLDAS:NLDAS_FORA0125_H_2_0_Tair","K"],
                    "SOLR":["Shortwave radiation flux","Langleys",
                    "NLDAS:NLDAS_FORA0125_H_2_0_SWdown","W m-2"],
                    "WIND": ["Wind Speed", "m s-1",
                             "NLDAS:NLDAS_FORA0125_H_2_0_Wind_N",
                             "NLDAS:NLDAS_FORA0125_H_2_0_Wind_E", "m s-1"], #NLDAS provides wind speed in two components. I will need to calculate the resultant wind speed from these two components.
                    "CLOU":["Cloud Cover Estimate","Cloud Fraction", "NLDAS:NLDAS_FORA0125_H_2_0_SWdown","W m-2"]}

#If you add more constituents, you will need to expand this dict


In [ ]:
help(tsget)

In [ ]:
import numpy as np
import math

def nldas_dewpoint_timeseries_from_specific_humidity(aSpecHumidTSer, aAirTempTSer, aSource, aPressure=1013.25):
    """
    Compute dewpoint temperature timeseries from specific humidity and air temperature timeseries.

    Parameters:
        aSpecHumidTSer: Timeseries of specific humidity (kg/kg)
        aAirTempTSer: Timeseries of air temperature (deg F)
        aSource: Timeseries source object
        aPressure: Pressure in millibar (default: 1013.25)
    Returns:
        Dewpoint Temperature timeseries at the same times as aSpecHumidTSer (deg F)
    """
    # lTSGroup = [aSpecHumidTSer, aAirTempTSer]  # grouped for reference in comments, not strictly needed in Python
    lDewpointTS = atcTimeseries(aSource)
    copy_base_attributes(aSpecHumidTSer, lDewpointTS)
    lDewpointTS.Attributes['Constituent'] = 'DEWP'
    lDewpointTS.Attributes['TSTYPE'] = 'DEWP'
    lDewpointTS.Attributes['Scenario'] = 'COMPUTED'
    lDewpointTS.Attributes['Description'] = "Dewpoint temperature (Deg F) computed from Specific Humidity (kg/kg)"
    lDewpointTS.Attributes['SPCHUM'] = str(aSpecHumidTSer)
    # lDewpointTS.Dates = merge_dates(lTSGroup)
    lDewpointTS.Dates = aSpecHumidTSer.Dates
    lDewpointTS.numValues = aSpecHumidTSer.numValues

    lSpecHumidIndex = 0
    lAirTempIndex = 0
    lPoint = aSpecHumidTSer.Attributes.get('point', False)
    # Provide a 1-based to 0-based mapping if your objects need it
    # The .Values approach in VB likely needs to be a list or numpy array in Python

    for lValueIndex in range(1, lDewpointTS.numValues + 1):  # 1-based like VB

        lSpecHumidIndex = find_date_at_or_after(
            aSpecHumidTSer.Dates.Values,
            lDewpointTS.Dates.Value(lValueIndex),
            lSpecHumidIndex
        )
        lAirTempIndex = find_date_at_or_after(
            aAirTempTSer.Dates.Values,
            lDewpointTS.Dates.Value(lValueIndex),
            lAirTempIndex
        )

        if lPoint:
            lDate = j2date(lDewpointTS.Dates.Value(lValueIndex))
        else:
            lDate = j2date(lDewpointTS.Dates.Value(lValueIndex - 1))

        lAirTempC = (aAirTempTSer.Value(lAirTempIndex) - 32.0) * 5.0 / 9.0
        es = 6.112 * math.exp((17.67 * lAirTempC) / (lAirTempC + 243.5))
        e = aSpecHumidTSer.Value(lSpecHumidIndex) * aPressure / (0.378 * aSpecHumidTSer.Value(lSpecHumidIndex) + 0.622)
        RelHumid = e / es
        if RelHumid > 1.0:
            RelHumid = 1.0
        elif RelHumid < 0:
            RelHumid = 0.0
        lDewTempC = lAirTempC - ((100 - 100 * RelHumid) / 5)
        lDewpointTS.Value[lValueIndex] = (lDewTempC * 9.0 / 5.0) + 32.0

    return lDewpointTS



In [ ]:
CloudCoverTimeseriesFromSolar(47.629605, 0.5, 1, 31)

# Define the list of stations and constituents to download, and the details for each constituent

In [10]:
StationList=[['Seattle',47.629605, -122.348941,-8],
            ['Allahabad', 25.43,81.84,5.5],
            ['Asmara',15.33, 38.92,3]]


In [11]:
import pandas as pd
import numpy as np
import os
from tkinter import Tk, filedialog

# Function to validate latitude and longitude
def validate_lat_lon(lat, lon):
    try:
        lat = float(lat)
        lon = float(lon)
        if not (-90 <= lat <= 90):
            return False
        if not (-180 <= lon <= 180):
            return False
        return True
    except Exception:
        return False

# Prompt user for input method
print("How would you like to enter station data?")
print("1. Manual entry")
print("2. Upload from spreadsheet (CSV or XLSX)")
method = input("Enter 1 or 2: ").strip()

StationList = []

if method == '1':
    print("Enter station data as: StationName, Latitude, Longitude, TimeZoneAdjustment")
    print("Example: Seattle, 47.629605, -122.348941, -8")
    print("Enter a blank line to finish.")
    while True:
        entry = input("Station: ").strip()
        if not entry:
            break
        parts = [x.strip() for x in entry.split(',')]
        if len(parts) != 4:
            print("Invalid format. Please enter 4 values separated by commas.")
            continue
        name, lat, lon, tz = parts
        if not validate_lat_lon(lat, lon):
            print("Invalid latitude or longitude. Latitude must be -90 to 90, longitude -180 to 180.")
            continue
        try:
            StationList.append([name, float(lat), float(lon), float(tz)])
        except Exception:
            print("Invalid entry. Please check your values.")
elif method == '2':
    print("Please select your CSV or XLSX file with columns: StationName, Latitude, Longitude, TimeZoneAdjustment")
    Tk().withdraw()  # Hide the root window
    file_path = filedialog.askopenfilename(filetypes=[("CSV files", "*.csv"), ("Excel files", "*.xlsx;*.xls")])
    if file_path:
        if file_path.lower().endswith('.csv'):
            df = pd.read_csv(file_path)
        else:
            df = pd.read_excel(file_path)
        required_cols = ['StationName', 'Latitude', 'Longitude', 'TimeZoneAdjustment']
        if not all(col in df.columns for col in required_cols):
            print(f"Missing required columns. Found columns: {df.columns.tolist()}")
        else:
            for _, row in df.iterrows():
                if validate_lat_lon(row['Latitude'], row['Longitude']):
                    StationList.append([
                        str(row['StationName']),
                        float(row['Latitude']),
                        float(row['Longitude']),
                        float(row['TimeZoneAdjustment'])
                    ])
                else:
                    print(f"Invalid lat/lon for station {row['StationName']}. Skipping.")
    else:
        print("No file selected.")
else:
    print("Invalid selection. Please restart and choose 1 or 2.")

print("Final StationList:")
print(StationList)

How would you like to enter station data?
1. Manual entry
2. Upload from spreadsheet (CSV or XLSX)
Please select your CSV or XLSX file with columns: StationName, Latitude, Longitude, TimeZoneAdjustment
Final StationList:
[['CentralPark', 40.785091, -73.968285, -5.0], ['GoldenGate', 37.819929, -122.478255, -8.0], ['MillenniumPark', 41.882702, -87.619392, -6.0]]


In [12]:
from datetime import datetime

def get_date(prompt):
    while True:
        date_str = input(f"{prompt} (YYYY-MM-DD): ").strip()
        try:
            date_obj = datetime.strptime(date_str, "%Y-%m-%d")
            return date_obj.strftime("%Y-%m-%d")
        except ValueError:
            print("Invalid date format. Please enter the date as YYYY-MM-DD.")

print("Please enter the date range for data retrieval.")
start_date = get_date("Enter start date")
end_date = get_date("Enter end date")

print(f"Selected date range: {start_date} to {end_date}")

Please enter the date range for data retrieval.
Selected date range: 2015-01-01 to 2015-12-31


In [13]:

UStop = 49.3457868 # north lat
USleft = -124.7844079 # west long
USright = -66.9513812 # east long
USbottom =  24.7433195 # south lat
#US Lat and long coordinates


In [39]:
help(convertunitforHSPF)

Help on function convertunitforHSPF in module __main__:

convertunitforHSPF(constituent, df, LDAS_var)
    This function is for unit conversion



In [39]:
help(help(convertunitforHSPF))

Help on function convertunitforHSPF in module __main__:

convertunitforHSPF(constituent, df, LDAS_var)
    This function is for unit conversion



In [18]:
#Testing after moving to that directory
from cloud_cover_timeseries_from_solar import cloud_cover_timeseries_from_solar
WDMFileName='MetData_041626a.wdm'
wdm.createnewwdm(WDMFileName, overwrite=True)
index = 1
from datetime import datetime
with open("MetLog_041626a.txt", 'w') as Logfile:
    Logfile.write("Started Downloading the data at "
                + datetime.isoformat(datetime.now()) + " and saving in "
                + WDMFileName + "\n")
    for station in StationList:
        # Going through Each Station in the list
        TimeZoneAdjustment = station[3]
        Logfile.write("Station: " + station[0] + ", Latitude: " + str(station[1])
                        + ", Longitude: " + str(station[2])
                        + ", TimeZoneAdjustment: " + str(TimeZoneAdjustment)
                        + "\n")
        lat_deg = station[1]
        for const in Constituent:
            # Going through each constituent
            if station[1] < UStop and station[1] > USbottom and \
                station[2] > USleft and station[2] < USright:
                # NLDAS
                print(df.head())
                df.info()
                if const == "WIND":
                    wind_n_var = NLDAS_ConstituentDetails["WIND"][2]
                    wind_e_var = NLDAS_ConstituentDetails["WIND"][3]
                    TimeStep = 1
                    print(f'{station[0]} is in contiguous USA (NLDAS)')
                    print(f"Downloading WindN: {wind_n_var} and WindE: {wind_e_var}")
                    df_n = tsget.ldas(lat=station[1], lon=station[2],
                                      variables=wind_n_var,
                                      startDate=start_date,
                                      endDate=end_date)
                    df_e = tsget.ldas(lat=station[1], lon=station[2],
                                      variables=wind_e_var,
                                      startDate=start_date,
                                      endDate=end_date)
                    wind_speed = np.sqrt(df_n.iloc[:, 0] ** 2 + df_e.iloc[:, 0] ** 2)
                    wind_speed.name = "WIND"
                    df = wind_speed
                    column_name = "WIND"
                elif const == "CLOU": #start here
                    TimeStep = 1
                    solar_n_var = NLDAS_ConstituentDetails["CLOU"][2]
                    df = tsget.ldas(lat=station[1], lon=station[2],variables=solar_n_var,
                                    startDate=start_date,
                                    endDate=end_date)
                    #df=df*3
                    column_name = "CLOU"
                    solar_n_var = NLDAS_ConstituentDetails["CLOU"][2]
                    TimeStep = 1

                    raw_df = tsget.ldas(
                    lat=station[1], lon=station[2],
                    variables=solar_n_var,
                    startDate=start_date, endDate=end_date
                     )

                    solr_series = raw_df.iloc[:, 0].dropna()
                    df = build_clou_from_solr(solr_series, lat_deg)
                    LDAS_variable = solar_n_var
                    column_name = "CLOU"

                #else:
                    #print(f"Downloading SOLR for Cloud Cover from: {solar_n_var} ")
                    #print(df.head())
                    #print(df.info())
                     #   df = process_cloud_cover_from_solr(df, #solr_col=NLDAS_ConstituentDetails["CLOU"][2], #lat_deg=lat_deg)
                else:
                    LDAS_variable = NLDAS_ConstituentDetails[const][2]
                    TimeStep = 1
                    print(f'{station[0]} is in contiguous USA')
                    print(LDAS_variable)
                    df = tsget.ldas(lat=station[1], lon=station[2],
                                   variables=LDAS_variable,
                                   startDate=start_date,
                                   endDate=end_date)
                    column_name = df.columns[0]
                    df = df[column_name]
                    df.dropna()
                    df = convertunitforHSPF(const, df, LDAS_variable)
                    elif const == "CLOU":
                        solar_n_var = NLDAS_ConstituentDetails["CLOU"][2]
                        df = tsget.ldas(lat=station[1], lon=station[2],variables=solar_n_var,
                                        startDate=start_date,
                                        endDate=end_date)
                        df=df*3
                        column_name = "CLOU"
                        print(f"Downloading SOLR for Cloud Cover from: {solar_n_var} ")
                        print(df.head())
                        print(df.info())

            else:
                # GLDAS
                if const == "WIND":
                    LDAS_variable = GLDAS_ConstituentDetails["WIND"][2]
                    TimeStep = 3
                    print(f'{station[0]} is outside contiguous USA (GLDAS)')
                    print(f"Downloading WIND: {LDAS_variable}")
                    df = tsget.ldas(lat=station[1], lon=station[2],
                                   variables=LDAS_variable,
                                   startDate=start_date,
                                   endDate=end_date)
                    column_name = df.columns[0]
                    df = df[column_name]
                    df.dropna()
                    df = convertunitforHSPF(const, df, LDAS_variable)
                else:
                    LDAS_variable = GLDAS_ConstituentDetails[const][2]
                    TimeStep = 3
                    print(f'{station} is outside contiguous USA')
                    print(LDAS_variable)
                    df = tsget.ldas(lat=station[1], lon=station[2],
                                   variables=LDAS_variable,
                                   startDate=start_date,
                                   endDate=end_date)
                    column_name = df.columns[0]
                    df = df[column_name]
                    df.dropna()
                    df = convertunitforHSPF(const, df, LDAS_variable)

            wdm.createnewdsn(WDMFileName, index,
                            constituent=const,
                            scenario="OBSERVED", location=station[0][0:8],
                            tcode=3, statid=station[0], tsstep=TimeStep,
                            description=LDAS_variable if const != "WIND" or station[1] >= UStop or station[1] <= USbottom or station[2] <= USleft or station[2] >= USright else "Vector Wind Speed")
            wdm.csvtowdm(WDMFileName, index, input_ts=df)
            Logfile.write("Constituent: " + const + ", Column Name:"
                + column_name + ", DSN: " + str(index) + "\n")
            index += 1

      StationName   Latitude   Longitude  TimeZoneAdjustment
0     CentralPark  40.785091  -73.968285                  -5
1      GoldenGate  37.819929 -122.478255                  -8
2  MillenniumPark  41.882702  -87.619392                  -6
<class 'pandas.DataFrame'>
RangeIndex: 3 entries, 0 to 2
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   StationName         3 non-null      str    
 1   Latitude            3 non-null      float64
 2   Longitude           3 non-null      float64
 3   TimeZoneAdjustment  3 non-null      int64  
dtypes: float64(2), int64(1), str(1)
memory usage: 228.0 bytes
CentralPark is in contiguous USA
NLDAS:NLDAS_FORA0125_H_2_0_Tair
Datetime:UTC
2015-01-01 00:00:00+00:00   -2.54
2015-01-01 01:00:00+00:00   -2.61
2015-01-01 02:00:00+00:00   -2.68
2015-01-01 03:00:00+00:00   -2.75
2015-01-01 04:00:00+00:00   -2.95
Freq: h, Name: NLDAS_FORA0125_H_2_0_Tair:K, dtype: float64
<cl

AttributeError: 'numpy.ndarray' object has no attribute 'strip'

In [34]:
help(tsget.ldas)

Help on function ldas_api in module tsgettoolbox.functions.ldas:

ldas_api(
    lat=None,
    lon=None,
    variables=None,
    startDate=None,
    endDate=None,
    variable=None
)
    global:grid:::Land Data Assimilation System, deprecated, use one of the ldas_* functions instead

    The time zone is always UTC.


    This command is deprecated. It has been split up into separate commands
    for each of the different LDAS products.  This command remains only so
    that old scripts don't break.

    Please use the following commands instead:
    ['ldas', 'ldas_gldas_noah', 'ldas_gldas_noah_v2_0', 'ldas_gldas_noah_v2_1', 'ldas_grace', 'ldas_merra', 'ldas_nldas_fora', 'ldas_nldas_noah', 'ldas_nldas_vic', 'ldas_smerge']




    +---------------------------+-------------+-----------+-----------+---------------+
    | Description/Name          | Spatial     | Lat Range | Lon Range | Time          |
    +===========================+=============+===========+===========+===============+
 

In [ ]:
import os
from shutil import move

def move_to_cwd(filename):
    cwd = os.getcwd()
    dest = os.path.join(cwd, os.path.basename(filename))
    if os.path.abspath(filename) != os.path.abspath(dest):
        try:
            move(filename, dest)
            print(f"Moved {filename} to {dest}")
        except Exception as e:
            print(f"Could not move {filename}: {e}")
    else:
        print(f"{filename} is already in the current working directory.")

# Move WDM and MetLog files to current working directory after creation
move_to_cwd(WDMFileName)
move_to_cwd(Logfile.name)

In [ ]:
import sys
sys.path.append(os.getcwd())
from cloud_cover_timeseries_from_solar import cloud_cover_timeseries_from_solar

def process_cloud_cover_from_pctsun(df, sun_col='PSUN'):
    """
    Update the 'CLOU' column in the DataFrame based on the percent sun column using cmp_cld.
    Assumes df has a DateTimeIndex and a column for percent sun (either 0-1 or 0-100 scale).
    """
    df['CLOU'] = cmp_cld(df[sun_col].values)
    return df

# Example usage for percent sun to cloud cover:
# dates = pd.date_range('2026-03-01', periods=5, freq='D')
# pct_sun = [80, 50, 20, 100, 0]
# df = pd.DataFrame({'PSUN': pct_sun, 'CLOU': np.nan}, index=dates)
# df = process_cloud_cover_from_pctsun(df, sun_col='PSUN')
# print(df)
